# 11.14 - Naive RAG (Build)

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Assemble everything from 11.1-11.13 into one end-to-end system: ingest -> split -> embed -> index -> retrieve -> generate. This is the integration moment - the baseline you can improve with advanced techniques.

## 2. Why Does This Matter?

A working baseline lets you measure improvement. Naive RAG handles a surprising share of real queries and teaches you where the ceiling is.

## 3. Prerequisites

Units 11.1-11.13.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build a complete naive RAG: in-memory corpus -> split -> embed -> Chroma -> retrieve k=4 -> llm()
- Test 3 questions and print context source + answer
- Establish a measurable baseline

## 5. Mental Model

Naive RAG is the simplest pipeline that works: documents in, answers out. It is the baseline every advanced technique is compared against.

```text
Corpus -> Split -> Embed -> Index -> Query -> Retrieve k -> Grounded Answer
```


## 6. Setup
Embedding helper + LLM helper, all fallback-safe so the notebook runs offline.

In [1]:
# Deterministic embedding helper.
# Loads all-MiniLM-L6-v2 if available; otherwise falls back to a hash-based
# vector so every cell still completes offline. The fallback still gives
# "similar text -> similar vector" behaviour via character-bigram overlap,
# so the demos remain meaningful without the model download.
import hashlib, numpy as np

_DIM = 384


def _hash_embed(texts):
    vecs = np.zeros((len(texts), _DIM))
    for i, t in enumerate(texts):
        bigrams = [t[j:j+2].lower() for j in range(len(t)-1)]
        for bg in bigrams:
            h = int(hashlib.md5(bg.encode()).hexdigest(), 16) % _DIM
            vecs[i, h] += 1.0
        norm = np.linalg.norm(vecs[i]) or 1.0
        vecs[i] = vecs[i] / norm
    return vecs


_model = None
_model_name = "all-MiniLM-L6-v2"


def get_embedder(force_fallback=False):
    """Return a function texts -> np.ndarray (N, dim)."""
    global _model
    if force_fallback:
        return _hash_embed
    if _model is None:
        try:
            from sentence_transformers import SentenceTransformer
            _model = SentenceTransformer(_model_name)
        except Exception as e:
            print("MiniLM unavailable, using hash fallback:", type(e).__name__)
            _model = None
    if _model is None:
        return _hash_embed
    return lambda texts: np.asarray(_model.encode(list(texts), convert_to_numpy=True))


def embed(texts, force_fallback=False):
    fn = get_embedder(force_fallback=force_fallback)
    return np.asarray(fn(texts), dtype=np.float32)


print("embedding dim:", _DIM)
print("backend:", _model_name if get_embedder() != _hash_embed else "hash-fallback")


embedding dim: 384


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4173.92it/s]

backend: all-MiniLM-L6-v2


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
import operator

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. In-Memory Corpus + Split
A small FAQ-style corpus. Chunk with RecursiveCharacterTextSplitter.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCS = [
    {"text": "Acme accepts returns within 30 days of purchase. Items must be in original "
             "packaging. Return shipping is paid by the customer unless the item is defective.",
     "source": "policy.txt"},
    {"text": "Standard shipping takes 5-7 business days. Express shipping arrives in 2-3 days "
             "for a flat fee. Free shipping on orders over $50 in the US.",
     "source": "shipping.txt"},
    {"text": "Refunds are issued to the original payment method within 5-7 business days after "
             "the returned item is received and inspected.",
     "source": "refunds.md"},
    {"text": "The warranty covers manufacturing defects for one year from purchase. Damage from "
             "misuse or accidents is not covered.",
     "source": "warranty.txt"},
]
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40,
                                          separators=["\n\n", "\n", ". ", " ", ""])
chunk_texts, chunk_ids, metas = [], [], []
for i, doc in enumerate(DOCS):
    for j, c in enumerate(splitter.split_text(doc["text"])):
        chunk_texts.append(c)
        chunk_ids.append(f"c{i}_{j}")
        metas.append({"source": doc["source"]})
print(f"{len(chunk_texts)} chunks across {len(DOCS)} docs")


4 chunks across 4 docs


## 8. Embed + Store in Chroma
Batch-embed and index with metadata.

In [4]:
import chromadb

emb = embed(chunk_texts)
client = chromadb.Client()
col = client.create_collection("naive_rag", embedding_function=None,
                               metadata={"hnsw:space": "cosine"})
col.add(documents=chunk_texts, embeddings=emb.tolist(), ids=chunk_ids, metadatas=metas)
print("indexed chunks:", col.count())


indexed chunks: 4


## 9. Define the Pipeline Functions
`retrieve(k=4)` returns top chunks with source metadata; `answer` builds the grounded prompt and calls llm(). The prompt instructs the model to answer ONLY from context and cite sources.

In [5]:
def retrieve(query, k=4):
    r = col.query(query_embeddings=embed([query]).tolist(), n_results=k)
    return list(zip(r["documents"][0], r["metadatas"][0], r["distances"][0]))


def build_context(hits):
    return "\n\n".join(f"[{i+1}] ({meta['source']})\n{doc}"
                         for i, (doc, meta, _) in enumerate(hits))


def answer(query, k=4):
    hits = retrieve(query, k)
    ctx = build_context(hits)
    prompt = (
        "Answer ONLY from the provided context. Cite sources as [i]. "
        "If the context doesn't answer, say you don't know.\n\n"
        f"Context:\n{ctx}\n\nQuestion: {query}\n\nAnswer:"
    )
    return hits, ctx, llm(prompt)


## 10. Test 3 Questions
For each question: retrieve, print the sources used, and print the grounded answer.

In [6]:
for q in ["How do I return a product I bought?",
          "How long does standard shipping take?",
          "Does the warranty cover accidental damage?"]:
    hits, ctx, ans = answer(q)
    print("=" * 70)
    print("Q:", q)
    print("Sources retrieved:")
    for doc, meta, dist in hits:
        print(f"   dist={dist:.3f} {meta['source']}: {doc[:48]}")
    print("ANSWER:", ans)
    print()


Q: How do I return a product I bought?
Sources retrieved:
   dist=0.514 policy.txt: Acme accepts returns within 30 days of purchase.
   dist=0.551 refunds.md: Refunds are issued to the original payment metho
   dist=0.828 warranty.txt: The warranty covers manufacturing defects for on
   dist=0.828 shipping.txt: Standard shipping takes 5-7 business days. Expre
ANSWER: To return a product you bought from Acme, follow these steps:

1. **Check the return window** – You must initiate the return within **30 days of purchase**.  
   [1]

2. **Prepare the item** – The product must be in its **original packaging** and in a condition that would be acceptable for resale.  
   [1]

3. **Arrange return shipping** –  
   * If the item is **not defective**, you are responsible for the return shipping cost.  
   * If the item is **defective**, Acme will cover the return shipping.  
   [1]

4. **Send the item back** – Ship the product to the return address provided by Acme (usually included on the retu

Q: How long does standard shipping take?
Sources retrieved:
   dist=0.217 shipping.txt: Standard shipping takes 5-7 business days. Expre
   dist=0.568 policy.txt: Acme accepts returns within 30 days of purchase.
   dist=0.685 refunds.md: Refunds are issued to the original payment metho
   dist=0.963 warranty.txt: The warranty covers manufacturing defects for on
ANSWER: Standard shipping takes 5‑7 business days. [1]



Q: Does the warranty cover accidental damage?
Sources retrieved:
   dist=0.243 warranty.txt: The warranty covers manufacturing defects for on
   dist=0.776 policy.txt: Acme accepts returns within 30 days of purchase.
   dist=0.840 refunds.md: Refunds are issued to the original payment metho
   dist=1.036 shipping.txt: Standard shipping takes 5-7 business days. Expre
ANSWER: No. The warranty only covers manufacturing defects; damage from misuse or accidents is not covered【1】.



## 11. Baseline Reflection
Record the sources each answer cited. If a question is answered well, retrieval found the right chunk - if not, retrieval/precision is your next lever (reranking, hybrid search, metadata).

In [7]:
for q in ["How do I return a product I bought?",
          "How long does standard shipping take?"]:
    hits, _, _ = answer(q)
    sources = sorted({m['source'] for _, m, _ in hits})
    print(f"{q[:42]:42s} -> sources: {sources}")


How do I return a product I bought?        -> sources: ['policy.txt', 'refunds.md', 'shipping.txt', 'warranty.txt']


How long does standard shipping take?      -> sources: ['policy.txt', 'refunds.md', 'shipping.txt', 'warranty.txt']



## Common Mistakes

- Skipping evaluation (building without measuring).
- Over-engineering before establishing a baseline.
- Not handling edge cases (empty docs, failed queries).
- Not logging retrieved context next to the answer.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Wrong answer but good retrieval | Prompt/context issue | Improve instructions, cite sources |
| Irrelevant chunks retrieved | Query/embedding/chunking | Rewrite query, hybrid search |
| Answer invents facts | Poor grounding | Require citations + abstention |
| Slow/expensive | Too many chunks/calls | Reduce k, batch embeddings |

## Best Practices

- Establish a naive baseline first.
- Evaluate retrieval separately from generation.
- Log retrieved context with each answer.
- Add one advanced technique at a time and measure.

## Hands-On Practice

1. **Basic:** Implement the NaiveRAG pipeline above.
2. **Guided:** Ingest 10 docs, test with 5 queries.
3. **Independent:** Build from scratch without the template.
4. **Realistic:** Ingest a real document set.
5. **Challenge:** Add error handling, logging, and evaluation.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
